# 05 — Age-only Cox PH: time until deterioration (baseline)

**Primary:** conditional median remaining years using the fitted Cox baseline hazard, without extrapolation. Missing medians are expected when survival stays above 50%.

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the original project directory.
PROJECT = Path.cwd().resolve()

if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

if PROJECT.name == "bridge_time_primary_notebooks":
    PROJECT = PROJECT.parent

# Define directories.
FINAL = PROJECT / "data" / "final"
RESULTS = PROJECT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

# Check which datasets exist.
print("Project directory:", PROJECT)
print("Available datasets:")
for file in sorted(FINAL.glob("*.parquet")):
    print(" -", file.name)

# Load training data.
train = pd.read_parquet(
    FINAL / "bridge_survival_train.parquet"
)

# Prefer aligned test data if available.
aligned_path = FINAL / "bridge_survival_test_aligned.parquet"
regular_path = FINAL / "bridge_survival_test.parquet"

if aligned_path.exists():
    test = pd.read_parquet(aligned_path)
    print("Using aligned test dataset")
elif regular_path.exists():
    test = pd.read_parquet(regular_path)
    print("Using regular test dataset")
else:
    raise FileNotFoundError("Neither test dataset was found.")

print("Training:", train.shape)
print("Testing:", test.shape)

display(train.head())

Project directory: /Users/annie/Downloads/bridge-deterioration
Available datasets:
 - bridge_historical_validation.parquet
 - bridge_survival_dataset.parquet
 - bridge_survival_test.parquet
 - bridge_survival_test_aligned.parquet
 - bridge_survival_train.parquet
Using aligned test dataset
Training: (498875, 14)
Testing: (171766, 15)


,bridge_id,start_year,bridge_age,YEAR_BUILT_027,YEAR_RECONSTRUCTED_106,ADT_029,STRUCTURE_KIND_043A,STRUCTURE_TYPE_043B,STRUCTURE_LEN_MT_049,end_year,event,stop_reason,last_year,duration
0,01_00000000000S700,1996,7.0,1989.0,0.0,80.0,5,01,11.0,1999.0,1,Deterioration,2005,3.0
1,01_00000000000S702,2003,4.0,1999.0,0.0,50.0,5,05,14.7,2020.0,0,Training cutoff,2025,17.0
2,01_00000000000S703,2006,4.0,2002.0,0.0,150.0,5,01,13.7,2020.0,0,Training cutoff,2025,14.0
3,01_0000000000M0022,1998,56.0,1942.0,0.0,800.0,1,02,43.3,2006.0,1,Deterioration,2025,8.0
4,01_00000BLN0BR0010,2012,38.0,1974.0,0.0,250.0,1,19,6.4,2015.0,1,Deterioration,2025,3.0


In [8]:

# Step 2: Check variables and missing values

columns = ["bridge_age", "duration", "event"]

print("Training data summary:")
print(train[columns].describe())

print("\nMissing values:")
print(train[columns].isnull().sum())

print("\nEvent distribution:")
print(train["event"].value_counts())

print("\nNon-positive durations:")
print((train["duration"] <= 0).sum())


Training data summary:
          bridge_age       duration          event
count  497426.000000  498875.000000  498875.000000
mean       20.174613      11.613126       0.475464
std        20.213338       8.605542       0.499398
min         0.000000       1.000000       0.000000
25%         3.000000       4.000000       0.000000
50%        15.000000       9.000000       0.000000
75%        31.000000      18.000000       1.000000
max       196.000000      28.000000       1.000000

Missing values:
bridge_age    1449
duration         0
event            0
dtype: int64

Event distribution:
event
0    261678
1    237197
Name: count, dtype: int64

Non-positive durations:
0


In [9]:

# Step 3: Preprocessing for Age-only Cox Model

# Calculate median from training data only
age_median = train["bridge_age"].median()

# Prepare training data
cox_train = train[["bridge_age", "duration", "event"]].copy()

# Fill missing bridge ages
cox_train["bridge_age"] = cox_train["bridge_age"].fillna(age_median)

# Check results
print("Median bridge age:", age_median)
print("Training records:", len(cox_train))
print("\nMissing values after preprocessing:")
print(cox_train.isnull().sum())

display(cox_train.head())


Median bridge age: 15.0
Training records: 498875

Missing values after preprocessing:
bridge_age    0
duration      0
event         0
dtype: int64


,bridge_age,duration,event
0,7.0,3.0,1
1,4.0,17.0,0
2,4.0,14.0,0
3,56.0,8.0,1
4,38.0,3.0,1


In [10]:

# Step 4: Fit Age-only Cox Proportional Hazards Model

from lifelines import CoxPHFitter

cox_model = CoxPHFitter()

cox_model.fit(
    cox_train,
    duration_col="duration",
    event_col="event"
)

cox_model.print_summary()


<lifelines.CoxPHFitter: fitted with 498875 total observations, 261678 right-censored observations>
             duration col = 'duration'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 498875
number of events observed = 237197
   partial log-likelihood = -2927561.48
         time fit was run = 2026-10-30 05:13:04 UTC

---
            coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                   
bridge_age  0.02      1.02      0.00            0.02            0.02                1.02                1.02

            cmp to      z      p  -log2(p)
covariate                                 
bridge_age    0.00 203.24 <0.005       inf
---
Concordance = 0.65
Partial AIC = 5855124.97
log-likelihood ratio test = 35379.98 on 1 df
-log2(p) of ll-ratio test = inf

## Primary output: conditional median remaining years

Preprocess the aligned test population before prediction. The Cox baseline hazard is a step function learned from training data. Where conditional survival does not fall to 50% before the final observed training duration, the median is **not estimable**; this is not a prediction of zero years. The full-population hazard-ranking C-index is separate from the estimated median time and does not measure error in predicted years. The short 2021–2025 test follow-up limits direct validation of long-term estimates.

In [11]:
# Prepare the test data BEFORE making time predictions.
# Use the median fitted on training data; do not learn preprocessing from test data.
cox_test = test[["bridge_id", "bridge_age", "entry_duration"]].copy()
cox_test["bridge_age"] = pd.to_numeric(cox_test["bridge_age"], errors="coerce").fillna(age_median)
cox_test["entry_duration"] = pd.to_numeric(cox_test["entry_duration"], errors="coerce")
assert cox_test["bridge_age"].notna().all()
assert cox_test["entry_duration"].notna().all()
assert (cox_test["entry_duration"] >= 0).all()
assert cox_test["bridge_id"].is_unique
print("Prepared test records:", len(cox_test))
print("Missing values:\n", cox_test.isna().sum())

Prepared test records: 171766
Missing values:
 bridge_id         0
bridge_age        0
entry_duration    0
dtype: int64


In [12]:
# Conditional median remaining time from the fitted Cox baseline hazard.
# For a bridge still Good at entry time a, solve S(a + m | x)/S(a | x) <= 0.5.
# This does NOT extrapolate the baseline hazard beyond observed training support.
from lifelines.utils import concordance_index

base_h = cox_model.baseline_cumulative_hazard_.iloc[:, 0]
times = base_h.index.to_numpy(dtype=float)
values = base_h.to_numpy(dtype=float)
entry = cox_test["entry_duration"].to_numpy(dtype=float)
ph = cox_model.predict_partial_hazard(cox_test[["bridge_age"]]).to_numpy().ravel()

idx0 = np.searchsorted(times, entry, side="right") - 1
Hentry = np.where(idx0 >= 0, values[np.maximum(idx0, 0)], 0.0)
target_hazard = Hentry + np.log(2) / ph
cross = np.searchsorted(values, target_hazard, side="left")

within_support = np.isfinite(entry) & (entry >= 0) & (entry <= times[-1])
estimable = within_support & np.isfinite(ph) & (ph > 0) & (cross < len(times))
median_total = np.full(len(entry), np.nan)
median_total[estimable] = times[cross[estimable]]
remaining = np.where(estimable, np.maximum(median_total - entry, 0), np.nan)

primary_predictions = test[["bridge_id", "entry_duration", "test_duration", "event"]].copy()
primary_predictions["median_remaining_years"] = remaining
primary_predictions["median_total_years_since_first_good"] = median_total
primary_predictions["median_estimable"] = estimable
primary_predictions["within_training_time_support"] = within_support
primary_predictions["risk_score"] = ph
primary_predictions.to_csv(RESULTS / "time_cox_age_only.csv", index=False)

print("Estimable median remaining time:", int(estimable.sum()), "/", len(estimable))
print("Entry within training support:", int(within_support.sum()), "/", len(within_support))
print(primary_predictions["median_remaining_years"].describe().to_string())

# This is an exploratory discrimination metric on the FULL aligned test set.
# It is not an error-in-years measure, and the short test follow-up limits evaluation.
valid = np.isfinite(ph) & np.isfinite(test["test_duration"].to_numpy(dtype=float))
if valid.sum() > 2 and test.loc[valid, "event"].sum() > 0:
    c_index_full = concordance_index(
        test.loc[valid, "test_duration"],
        -ph[valid],  # higher hazard implies shorter survival
        test.loc[valid, "event"]
    )
    print("Full aligned test C-index (Cox hazard ranking):", round(c_index_full, 4))
print("Note: median time unavailable means 50% deterioration was not reached within model support.")

Estimable median remaining time: 45235 / 171766
Entry within training support: 130088 / 171766
count    45235.000000
mean        19.312103
std          5.523270
min          1.000000
25%         15.000000
50%         21.000000
75%         24.000000
max         25.000000
Full aligned test C-index (Cox hazard ranking): 0.6013
Note: median time unavailable means 50% deterioration was not reached within model support.
